In [ ]:
class Node:
    """Represents a node in the search tree"""

    def __init__(self, position, parent=None, g=0, h=0):
        """Initialize a node for pathfinding

        Args:
            position (tuple): (x, y) coordinate
            parent (Node): Parent node in the path
            g (float): Cost from start to this node
            h (float): Heuristic cost from this node to goal (Manhattan distance)
        """
        # The (x, y) cell this node represents on the grid
        self.position = position

        # The node we came from (None if this is the start node)
        self.parent = parent

        # g: actual cost to reach this node from the start (number of steps taken so far)
        self.g = g

        # h: estimated cost from this node to the goal (Manhattan distance heuristic)
        # Manhattan distance = |x1 - x2| + |y1 - y2|, never overestimates → admissible
        self.h = h

        # f: total estimated cost of the path through this node
        # f = g + h → A* always expands the node with the lowest f first
        self.f = g + h

        # depth: how many steps from the root to this node
        # depth == timestep in a unit-cost grid (each move = 1 step = 1 time unit)
        # used for time-indexed collision checking (which robot is where at time t)
        self.depth = 0 if parent is None else parent.depth + 1

        # conflicts: list of collision dicts detected at this node
        # used by CBS (Conflict-Based Search) high-level to store collisions
        # so it knows which conflict to resolve when branching
        # each conflict looks like:
        # {
        #   'type'    : 'vertex' or 'edge',
        #   'robots'  : [r1, r2],         ← the two robots involved
        #   'location': (x, y),           ← where the conflict happened
        #   'timestep': int               ← when it happened
        # }
        self.conflicts = []

    def __lt__(self, other):
        # heapq (the priority queue used in A*) needs to compare nodes
        # primary sort: lower f wins → expand cheapest path first
        if self.f != other.f:
            return self.f < other.f
        # tie-break: if two nodes have the same f, prefer the one with lower h
        # lower h means closer to the goal → reaches solution faster
        return self.h < other.h

    def __eq__(self, other):
        # two nodes are the same if they represent the same grid cell
        # used by the 'visited' set in A* to avoid re-expanding the same position
        return self.position == other.position

    def __hash__(self):
        # makes Node hashable so it can be stored in sets and used as dict keys
        # the visited set in A* relies on this
        # hashing by position: two nodes at the same cell → same hash
        return hash(self.position)

    def reconstruct_path(self):
        """Reconstruct path from start to this node

        Returns:
            List[tuple]: Path as list of (x, y) coordinates
        """
        path = []
        current = self

        # walk backwards through parent pointers until we hit the root (parent = None)
        while current is not None:
            path.append(current.position)
            current = current.parent

        # path is built in reverse (goal → start), so flip it
        return list(reversed(path))

    def has_conflict(self):
        # returns True if CBS detected any collision at this node
        # CBS high-level checks this to decide if a solution is valid
        # or if it needs to branch and add constraints to resolve the conflict
        return len(self.conflicts) > 0

    def add_conflict(self, conflict):
        """Register a conflict at this node (used by CBS high-level)

        Args:
            conflict (dict): {
                'type'    : 'vertex' | 'edge',
                'robots'  : [r1, r2],
                'location': (x,y) or ((x1,y1),(x2,y2)),
                'timestep': int
            }
        """
        # append the conflict dict to the list
        # CBS will later read conflicts[0] to decide which one to resolve first
        self.conflicts.append(conflict)

print("Class Node defined")